In [1]:

# %% [markdown]
"""
# Predictive Maintenance with Window-Based Feature Aggregation and Pressure Classification

**Objective:**  
Build predictive models for:
- **mech_1**: Predict fault (0/1) between sensors A and C.
- **mech_2**: Predict fault (0/1) between sensors C and B.
- **Pressure**: Instead of predicting a continuous value, classify the pressure as:
    - 0: "low" (pressure < 4.9 bar),
    - 1: "high" (pressure > 5.2 bar), or
    - 2: "normal" (pressure between 4.9 and 5.2 bar).

We use a sliding window of datapoints (e.g. window size = 10) to compute aggregated features from the "Time series" column.
"""
# %% 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# %% [markdown]
"""
## 1. Data Loading and Preprocessing

Load the four CSV files.  
Each CSV file (if no header is present) is assumed to have 3 columns:
- **Time series**: Time (in milliseconds) of the workflow.
- **Pressure**: Measured pressure (continuous, in bar).
- **Fault**: Binary indicator (0 or 1) indicating the presence of a mechanical fault.

The files correspond to different sensor configurations:
- **AC** and **CA**: Their fault column is used for predicting **mech_1** (fault between sensors A and C).
- **CB** and **BC**: Their fault column is used for predicting **mech_2** (fault between sensors C and B).
"""
# %% 
# Adjust file paths as needed; assume CSV files do not have headers.
ac_df = pd.read_csv('C:/Users/Алишер/Desktop/NazarbayevUniversity/DatasetPdM/Class_regres_dataset/AC_Down_to_Sort - Sheet1.csv', header=0)
ca_df = pd.read_csv('C:/Users/Алишер/Desktop/NazarbayevUniversity/DatasetPdM/Class_regres_dataset/CA_Sort_to_Down - Sheet1.csv', header=0)
cb_df = pd.read_csv('C:/Users/Алишер/Desktop/NazarbayevUniversity/DatasetPdM/Class_regres_dataset/CB_Sort_to_Upstream - Sheet1.csv', header=0)
bc_df = pd.read_csv('C:/Users/Алишер/Desktop/NazarbayevUniversity/DatasetPdM/Class_regres_dataset/BC_Upstream_to_Sort - Sheet1.csv', header=0)

# If files have extra columns, select only the first 3:
ac_df = ac_df.iloc[:, :3]
ca_df = ca_df.iloc[:, :3]
cb_df = cb_df.iloc[:, :3]
bc_df = bc_df.iloc[:, :3]

# Assign column names
col_names = ['Time series', 'Pressure', 'Fault']
ac_df.columns = col_names
ca_df.columns = col_names
cb_df.columns = col_names
bc_df.columns = col_names

# Add sensor configuration labels
ac_df['sensor_config'] = 'AC'
ca_df['sensor_config'] = 'CA'
cb_df['sensor_config'] = 'CB'
bc_df['sensor_config'] = 'BC'

In [2]:
# %% [markdown]
"""
## 2. Create Window-Based Aggregated Features

Define a function that converts a time series (or a column) into aggregated features (mean, std, min, max, last) over a sliding window.
"""
# %% 
def create_window_features(df, window_size=10, feature_column='Time series', label_column=None):
    windows = []
    labels = []
    for i in range(len(df) - window_size + 1):
        window = df.iloc[i:i+window_size]
        feat_mean = window[feature_column].mean()
        feat_std  = window[feature_column].std()
        feat_min  = window[feature_column].min()
        feat_max  = window[feature_column].max()
        feat_last = window[feature_column].iloc[-1]
        window_feat = {
            f'{feature_column}_mean': feat_mean,
            f'{feature_column}_std': feat_std,
            f'{feature_column}_min': feat_min,
            f'{feature_column}_max': feat_max,
            f'{feature_column}_last': feat_last
        }
        windows.append(window_feat)
        if label_column is not None:
            # For classification, take the label from the last row of the window.
            labels.append(window[label_column].iloc[-1])
    df_window = pd.DataFrame(windows)
    if label_column is not None:
        df_window[label_column] = labels
    return df_window

# %% [markdown]
"""
### Prepare Datasets for Each Task

**mech_1 Classification:** Combine AC and CA files.  
**mech_2 Classification:** Combine CB and BC files.  
**Pressure Classification:** Combine all files and transform the Pressure target to a categorical one.
"""
# %% 
# For mech_1:
df_mech1 = pd.concat([ac_df, ca_df], axis=0).copy()
df_mech1_window = create_window_features(df_mech1, window_size=10, feature_column='Time series', label_column='Fault')
df_mech1_window.rename(columns={'Fault': 'mech_1'}, inplace=True)

# For mech_2:
df_mech2 = pd.concat([cb_df, bc_df], axis=0).copy()
df_mech2_window = create_window_features(df_mech2, window_size=10, feature_column='Time series', label_column='Fault')
df_mech2_window.rename(columns={'Fault': 'mech_2'}, inplace=True)

# For Pressure: Combine all files
df_pressure = pd.concat([ac_df, ca_df, cb_df, bc_df], axis=0).copy()
df_pressure_window = create_window_features(df_pressure, window_size=10, feature_column='Time series', label_column='Pressure')


In [3]:
# %% [markdown]
"""
### Transform Pressure Target

Instead of predicting a continuous value, we convert the pressure target into categories:
- **0 ("low")**: Pressure < 4.9 bar
- **1 ("high")**: Pressure > 5.2 bar
- **2 ("normal")**: Pressure within 4.9 to 5.2 bar
"""
# %% 
def transform_pressure(p):
    if p < 4.9:
        return 0   # low
    elif p > 5.2:
        return 1   # high
    else:
        return 2   # normal

# Apply the transformation to the pressure window dataset
df_pressure_window['pressure_class'] = df_pressure_window['Pressure'].apply(transform_pressure)
# For the classification model, we drop the original continuous Pressure column.
df_pressure_window = df_pressure_window.drop(columns=['Pressure'])

In [4]:
# %% [markdown]
"""
## 3. Split the Data and Preprocess

Split each dataset (mech_1, mech_2, and Pressure) into training, validation, and test sets.
For Pressure classification, use the transformed target.
"""
# %% 
# mech_1 Classification
X_mech1 = df_mech1_window.drop(columns=['mech_1'])
y_mech1 = df_mech1_window['mech_1']
X_mech1_train, X_mech1_temp, y_mech1_train, y_mech1_temp = train_test_split(
    X_mech1, y_mech1, test_size=0.3, random_state=42, stratify=y_mech1)
X_mech1_val, X_mech1_test, y_mech1_val, y_mech1_test = train_test_split(
    X_mech1_temp, y_mech1_temp, test_size=0.5, random_state=42)

# mech_2 Classification
X_mech2 = df_mech2_window.drop(columns=['mech_2'])
y_mech2 = df_mech2_window['mech_2']
X_mech2_train, X_mech2_temp, y_mech2_train, y_mech2_temp = train_test_split(
    X_mech2, y_mech2, test_size=0.3, random_state=42, stratify=y_mech2)
X_mech2_val, X_mech2_test, y_mech2_val, y_mech2_test = train_test_split(
    X_mech2_temp, y_mech2_temp, test_size=0.5, random_state=42)

# Pressure Classification
X_pressure = df_pressure_window.drop(columns=['pressure_class'])
y_pressure = df_pressure_window['pressure_class']
X_pressure_train, X_pressure_temp, y_pressure_train, y_pressure_temp = train_test_split(
    X_pressure, y_pressure, test_size=0.3, random_state=42, stratify=y_pressure)
X_pressure_val, X_pressure_test, y_pressure_val, y_pressure_test = train_test_split(
    X_pressure_temp, y_pressure_temp, test_size=0.5, random_state=42)

# Preprocessing pipelines
clf_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
reg_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Scale features for classification tasks (mech_1 & mech_2 use the same pipeline)
X_mech1_train_scaled = clf_preprocessor.fit_transform(X_mech1_train)
X_mech1_val_scaled = clf_preprocessor.transform(X_mech1_val)
X_mech1_test_scaled = clf_preprocessor.transform(X_mech1_test)

X_mech2_train_scaled = clf_preprocessor.fit_transform(X_mech2_train)
X_mech2_val_scaled = clf_preprocessor.transform(X_mech2_val)
X_mech2_test_scaled = clf_preprocessor.transform(X_mech2_test)

# For pressure classification
X_pressure_train_scaled = reg_preprocessor.fit_transform(X_pressure_train)
X_pressure_val_scaled = reg_preprocessor.transform(X_pressure_val)
X_pressure_test_scaled = reg_preprocessor.transform(X_pressure_test)


In [5]:
#%% [markdown]
"""
## 4. Model Training and Evaluation

Train models for each task using Random Forest Classifiers.
"""
# %% 
# Train and evaluate mech_1 classifier
mech1_clf = RandomForestClassifier(n_estimators=100, random_state=42)
mech1_clf.fit(X_mech1_train_scaled, y_mech1_train)
y_mech1_val_pred = mech1_clf.predict(X_mech1_val_scaled)
print("Validation results for mech_1 classifier:")
print("Accuracy:", accuracy_score(y_mech1_val, y_mech1_val_pred))
print(classification_report(y_mech1_val, y_mech1_val_pred))

# Train and evaluate mech_2 classifier
mech2_clf = RandomForestClassifier(n_estimators=100, random_state=42)
mech2_clf.fit(X_mech2_train_scaled, y_mech2_train)
y_mech2_val_pred = mech2_clf.predict(X_mech2_val_scaled)
print("Validation results for mech_2 classifier:")
print("Accuracy:", accuracy_score(y_mech2_val, y_mech2_val_pred))
print(classification_report(y_mech2_val, y_mech2_val_pred))

# Train and evaluate pressure classifier
pressure_clf = RandomForestClassifier(n_estimators=100, random_state=42)
pressure_clf.fit(X_pressure_train_scaled, y_pressure_train)
y_pressure_val_pred = pressure_clf.predict(X_pressure_val_scaled)
print("Validation results for pressure classifier:")
print("Accuracy:", accuracy_score(y_pressure_val, y_pressure_val_pred))
print(classification_report(y_pressure_val, y_pressure_val_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_pressure_val, y_pressure_val_pred))


Validation results for mech_1 classifier:
Accuracy: 0.968
              precision    recall  f1-score   support

           0       0.96      0.99      0.97        74
           1       0.98      0.94      0.96        51

    accuracy                           0.97       125
   macro avg       0.97      0.96      0.97       125
weighted avg       0.97      0.97      0.97       125

Validation results for mech_2 classifier:
Accuracy: 0.968
              precision    recall  f1-score   support

           0       0.99      0.96      0.97        74
           1       0.94      0.98      0.96        51

    accuracy                           0.97       125
   macro avg       0.96      0.97      0.97       125
weighted avg       0.97      0.97      0.97       125

Validation results for pressure classifier:
Accuracy: 0.9563492063492064
              precision    recall  f1-score   support

           0       0.96      0.93      0.94        96
           1       0.97      1.00      0.98     

In [6]:
# %% [markdown]
"""
## 5. Deployment Function

The deployment function now accepts a **window** of time series values (a list or array) and a sensor configuration.
It computes aggregated features from the window and returns predictions:
- **mech_1_prediction** (if sensor configuration is AC or CA)
- **mech_2_prediction** (if sensor configuration is CB or BC)
- **pressure_prediction**: a categorical prediction (0 = low, 1 = high, 2 = normal)
"""
# %% 
def aggregate_window(window, feature_name='Time series'):
    window = np.array(window)
    return {
        f'{feature_name}_mean': window.mean(),
        f'{feature_name}_std': window.std(),
        f'{feature_name}_min': window.min(),
        f'{feature_name}_max': window.max(),
        f'{feature_name}_last': window[-1]
    }

def predict_system_status(window, sensor_config):
    """
    Parameters:
      - window: List or array of time series values (length should be at least the window size used in training, e.g., 10).
      - sensor_config: A string (one of 'AC', 'CA', 'CB', 'BC').
      
    Returns:
      Dictionary with:
         'mech_1_prediction' (if applicable),
         'mech_2_prediction' (if applicable),
         'pressure_prediction' (0 for low, 1 for high, 2 for normal).
    """
    features_dict = aggregate_window(window, feature_name='Time series')
    input_df = pd.DataFrame([features_dict])
    
    # Preprocess for classification (both mech_1 and mech_2, and pressure use similar windowed features)
    input_scaled_clf = clf_preprocessor.transform(input_df)
    input_scaled_reg = reg_preprocessor.transform(input_df)
    
    predictions = {}
    
    if sensor_config.upper() in ['AC', 'CA']:
        pred_mech1 = mech1_clf.predict(input_scaled_clf)[0]
        predictions['mech_1_prediction'] = int(pred_mech1)
    else:
        predictions['mech_1_prediction'] = None
    
    if sensor_config.upper() in ['CB', 'BC']:
        pred_mech2 = mech2_clf.predict(input_scaled_clf)[0]
        predictions['mech_2_prediction'] = int(pred_mech2)
    else:
        predictions['mech_2_prediction'] = None
    
    # Pressure prediction (categorical: 0=low, 1=high, 2=normal)
    pred_pressure = pressure_clf.predict(input_scaled_reg)[0]
    predictions['pressure_prediction'] = int(pred_pressure)
    
    return predictions

In [ ]:

# #example_window = [3799,3811,3692,3804,3805,3699,3805,3801,3698]
# # %% [markdown]
# """
# ## 6. Testing the Deployment Function

# Test predictions using an example window of time series values for different sensor configurations.
# """
# # %% 
# example_window = [3693,3808,3593,3700,3591,3692,3581]  # Example window (10 datapoints) 5/0
# for config in ['AC']:
#     pred = predict_system_status(example_window, config)
#     print(f"Input window (length {len(example_window)}), sensor_config = {config}:")
#     print(pred)
#     print()
# example_window = [3810,3804,3795,3804,3704,3914,3791]  # 5/1
# for config in ['CB']:
#     pred = predict_system_status(example_window, config)
#     print(f"Input window (length {len(example_window)}), sensor_config = {config}:")
#     print(pred)
#     print()

In [7]:
import json
import paho.mqtt.client as mqtt

# MQTT configuration
MQTT_BROKER = "test.mosquitto.org"
MQTT_PORT = 1883
MQTT_TOPIC = "mps512/feedback0202"

# Create and configure the MQTT client
mqtt_client = mqtt.Client()
mqtt_client.connect(MQTT_BROKER, MQTT_PORT, keepalive=60)

def send_output_to_node_red(output_dict):
    """
    Sends the output dictionary to Node-RED over MQTT.
    
    Parameters:
      output_dict (dict): A dictionary containing model output, for example:
                          {
                              "mech_1_prediction": 0,
                              "mech_2_prediction": 1,
                              "pressure_prediction": 2
                          }
    """
    # Convert dictionary to JSON string
    payload = json.dumps(output_dict)
    # Publish to the MQTT topic
    mqtt_client.publish(MQTT_TOPIC, payload)
    print(f"{payload}")

# Example usage:
# After obtaining the model output (for example, via your deployment function),
# call the following:
example_output = {
    "mech_1": 0,
    # "mech_2": 1,
    # "pressure": 2
}

send_output_to_node_red(example_output)
# import json
# import paho.mqtt.client as mqtt

# # MQTT configuration
# MQTT_BROKER = "test.mosquitto.org"
# MQTT_PORT = 1883
# MQTT_TOPIC = "mps512/feedback0202"

# # Create and configure the MQTT client
# mqtt_client = mqtt.Client()
# mqtt_client.connect(MQTT_BROKER, MQTT_PORT, keepalive=60)

# def send_output_to_node_red(output_dict):
#     """
#     Sends the output dictionary to Node-RED over MQTT in the required format:
#     {"mech_1": output, "mech_2": output, "pressure_status": output}
#     """
#     payload = json.dumps(output_dict)
#     mqtt_client.publish(MQTT_TOPIC, payload)
#     print(f"Published to topic {MQTT_TOPIC}: {payload}")

# # --- Example for sensor configuration "AC" ---
# # Assume our deployment function predict_system_status(window, sensor_config)
# # returns a dictionary like {"mech_1_prediction": 0, "mech_2_prediction": None, "pressure_prediction": 2}

# example_window_AC = [3693, 3808, 3593, 3700, 3591, 3692, 3581]
# predictions_ac = predict_system_status(example_window_AC, 'AC')
# # Reformat the dictionary to match our required output keys.
# output_data_ac = {
#     "mech_1": predictions_ac.get("mech_1_prediction", None),
#     # "mech_2": predictions_ac.get("mech_2_prediction", None),
#     "pressure_status": predictions_ac.get("pressure_prediction", None)
# }
# send_output_to_node_red(output_data_ac)

# # --- Example for sensor configuration "CB" ---
# example_window_CB = [3810, 3804, 3795, 3804, 3704, 3914, 3791]
# predictions_cb = predict_system_status(example_window_CB, 'CB')
# output_data_cb = {
#     # "mech_1": predictions_cb.get("mech_1_prediction", None),
#     "mech_2": predictions_cb.get("mech_2_prediction", None),
#     "pressure_status": predictions_cb.get("pressure_prediction", None)
# }
# send_output_to_node_red(output_data_cb)


{"mech_1": 0}


In [8]:
# import json
# import paho.mqtt.client as mqtt

# # MQTT configuration
# MQTT_BROKER = "test.mosquitto.org"
# MQTT_PORT = 1883
# MQTT_TOPIC = "mps512/feedback0202"

# # Create and configure the MQTT client
# mqtt_client = mqtt.Client()
# mqtt_client.connect(MQTT_BROKER, MQTT_PORT, keepalive=60)

# def send_output_to_node_red(output_dict):
#     """
#     Sends the output dictionary to Node-RED over MQTT.
    
#     Parameters:
#       output_dict (dict): A dictionary containing model output, for example:
#                           {
#                               "mech_1_prediction": 0,
#                               "mech_2_prediction": 1,
#                               "pressure_prediction": 2
#                           }
#     """
#     # Convert dictionary to JSON string
#     payload = json.dumps(output_dict)
#     # Publish to the MQTT topic
#     mqtt_client.publish(MQTT_TOPIC, payload)
#     print(f"{payload}")

# # Example usage:
# # After obtaining the model output (for example, via your deployment function),
# # call the following:
# example_output = {
#     "mech_1": 0,
#     "mech_2": 1,
#     "pressure": 2
# }

# send_output_to_node_red(example_output)
# # import json
# # import paho.mqtt.client as mqtt

# # # MQTT configuration
# # MQTT_BROKER = "test.mosquitto.org"
# # MQTT_PORT = 1883
# # MQTT_TOPIC = "mps512/feedback0202"

# # # Create and configure the MQTT client
# # mqtt_client = mqtt.Client()
# # mqtt_client.connect(MQTT_BROKER, MQTT_PORT, keepalive=60)

# # def send_output_to_node_red(output_dict):
# #     """
# #     Sends the output dictionary to Node-RED over MQTT in the required format:
# #     {"mech_1": output, "mech_2": output, "pressure_status": output}
# #     """
# #     payload = json.dumps(output_dict)
# #     mqtt_client.publish(MQTT_TOPIC, payload)
# #     print(f"Published to topic {MQTT_TOPIC}: {payload}")

# # # --- Example for sensor configuration "AC" ---
# # # Assume our deployment function predict_system_status(window, sensor_config)
# # # returns a dictionary like {"mech_1_prediction": 0, "mech_2_prediction": None, "pressure_prediction": 2}

# # example_window_AC = [3693, 3808, 3593, 3700, 3591, 3692, 3581]
# # predictions_ac = predict_system_status(example_window_AC, 'AC')
# # # Reformat the dictionary to match our required output keys.
# # output_data_ac = {
# #     "mech_1": predictions_ac.get("mech_1_prediction", None),
# #     # "mech_2": predictions_ac.get("mech_2_prediction", None),
# #     "pressure_status": predictions_ac.get("pressure_prediction", None)
# # }
# # send_output_to_node_red(output_data_ac)

# # # --- Example for sensor configuration "CB" ---
# # example_window_CB = [3810, 3804, 3795, 3804, 3704, 3914, 3791]
# # predictions_cb = predict_system_status(example_window_CB, 'CB')
# # output_data_cb = {
# #     # "mech_1": predictions_cb.get("mech_1_prediction", None),
# #     "mech_2": predictions_cb.get("mech_2_prediction", None),
# #     "pressure_status": predictions_cb.get("pressure_prediction", None)
# # }
# # send_output_to_node_red(output_data_cb)

import json
import paho.mqtt.client as mqtt

# MQTT configuration
MQTT_BROKER = "test.mosquitto.org"
MQTT_PORT = 1883
MQTT_TOPIC = "mps512/feedback0202"

# Создаём и подключаем MQTT‑клиент
mqtt_client = mqtt.Client()
mqtt_client.connect(MQTT_BROKER, MQTT_PORT, keepalive=60)

def send_output_to_node_red(output_dict):
    """
    Фильтрует None‑значения и отправляет JSON в Node-RED через MQTT.
    Окончательный формат JSON будет таким:
      {"mech_1": ..., "mech_2": ..., "pressure_status": ...}
    где ключей с None не будет.
    """
    # Переименование ключей и фильтрация None
    mapped = {
        "mech_1": output_dict.get("mech_1_prediction"),
        "mech_2": output_dict.get("mech_2_prediction"),
        "pressure_status": output_dict.get("pressure_prediction")
    }
    filtered = {k: v for k, v in mapped.items() if v is not None}

    # Конвертируем в JSON
    payload = json.dumps(filtered)
    
    # Публикуем
    mqtt_client.publish(MQTT_TOPIC, payload)
    print(f"Published to topic {MQTT_TOPIC}: {payload}")

# Пример использования:
example_output_ac = predict_system_status([3693,3808,3593,3700,3591,3692,3581], 'AC')
send_output_to_node_red(example_output_ac)

example_output_cb = predict_system_status([3810,3804,3795,3804,3704,3914,3791], 'CB')
send_output_to_node_red(example_output_cb)



Published to topic mps512/feedback0202: {"mech_1": 1, "pressure_status": 0}
Published to topic mps512/feedback0202: {"mech_2": 0, "pressure_status": 0}
